# Mako Reactor demo

Describe where the buttons go, and get back a stack of flat plates you can cut.

The stack is five plates bolted through the corners:

| layer | part | job |
| --- | --- | --- |
| 4 | `Backplate` | cosmetic top |
| 3 | `F1CapFaceplate` / `KeycapFaceplate` | button openings |
| 2 | `Switchplate` | 14x14 switch mounts |
| 1 | `WireSpaceModelU` | spacer for wiring and the USB-C breakout |
| 0 | `Base` | solid bottom |

In [1]:
import cadquery as cq
from affine import Affine
from cadquery import exporters
from IPython.display import SVG, display

import makoreactor as mr

in2mm = mr.in2mm

# Interactive OCP rendering, if jupyter-cadquery is installed.
try:
    from jupyter_cadquery import show as show_ocp
except ImportError:
    show_ocp = None


def show(*parts, width=1000, height=440, **kwargs):
    """Render parts, preferring the interactive OCP viewer.

    Falls back to a flat SVG preview when jupyter-cadquery is missing or no
    Jupyter server is reachable, so this notebook also runs headless.
    """
    if show_ocp is not None:
        try:
            return show_ocp(*parts, height=height, **kwargs)
        except Exception as exc:
            print(f"OCP viewer unavailable ({type(exc).__name__}); drawing SVG instead")

    for part in parts:
        if isinstance(part, cq.Assembly):
            shape = part.toCompound()
        elif isinstance(part, cq.Workplane):
            shape = part.val()
        else:
            shape = part
        display(SVG(exporters.getSVG(shape, {
            "width": width, "height": height, "marginLeft": 12, "marginTop": 12,
            "projectionDir": (0, 0, 1), "showAxes": False, "strokeWidth": 0.4,
        })))

Overwriting auto display for cadquery Workplane and Shape


Renders use [jupyter-cadquery](https://github.com/bernhard-42/jupyter-cadquery)
when it is available (`pip install jupyter-cadquery`), and fall back to flat SVG
otherwise. See the README for the OCP version pins.

## 1. Layouts

Six layouts ship with the library. They are all `Layout` subclasses, so any of
them can be passed as `layout=` to any part or assembly. Each also exists as a
ready-made instance: `mr.smashF1CapLayout`, `mr.smashGccmxLayout`,
`mr.smashKeycapLayout`, `mr.mako1HadoeLayout`, `mr.wideGcLayout`,
`mr.fgcF1CapLayout`.

`Mako1Hadoe` is `SquareCapLeverless` with the right-hand eight dropped by one
row pitch (22mm). In the stock layout the *bottom* row of the eight lines up
with the four directionals; in `Mako1Hadoe` the *top* row does, so home row sits
under the fingers.

`WideGc` keeps the smash spacing and the right hand's eight, and changes the
other three groups. The right thumb's five-button c-stick is cut back to its
bottom two — the pair the left thumb already has — so both thumbs get the same
two buttons and the same reach. The centre is Crane's, inherited from `Gccmx`:
a row of three where the stock layout has one. And the left hand loses its
pinky, the outermost of its four. Eighteen buttons, and with nothing off the
right hand the cluster ends up 13mm right of centre — `layout_affine` puts it
back if the plate needs it.

The smash layouts are drawn for a 14x6" plate and the FGC layout for a smaller
one, so each is shown on the plate it was designed around.

In [2]:
LAYOUTS = {
    #                     layout                  faceplate            plate (in)
    "CircleCapLeverless": (mr.CircleCapLeverless, mr.F1CapFaceplate,   (14, 6)),
    "Gccmx":              (mr.Gccmx,              mr.F1CapFaceplate,   (14, 6)),
    "SquareCapLeverless": (mr.SquareCapLeverless, mr.KeycapFaceplate,  (14, 6)),
    "Mako1Hadoe":         (mr.Mako1Hadoe,         mr.KeycapFaceplate,  (14, 6)),
    "WideGc":             (mr.WideGc,             mr.F1CapFaceplate,   (14, 6)),
    "FgcLeverless":       (mr.FgcLeverless,       mr.F1CapFaceplate,   (11.8, 6)),
}

for name, (layout_cls, faceplate_cls, (plate_w, plate_h)) in LAYOUTS.items():
    layout = layout_cls()
    faceplate = faceplate_cls(width=in2mm(plate_w), height=in2mm(plate_h), layout=layout)

    print(f'{name:20} {len(layout):3} buttons   {plate_w}x{plate_h}"')
    show(faceplate.generate(), names=[name])

CircleCapLeverless    20 buttons   14x6"
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

Gccmx                 22 buttons   14x6"
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

SquareCapLeverless    20 buttons   14x6"
+


...

CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

WideGc                18 buttons   14x6"
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

FgcLeverless          15 buttons   11.8x6"
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

The plate renders show the shape; this shows the change, group by group,
against the stock smash layout.


In [3]:
stock, wide = mr.CircleCapLeverless(), mr.WideGc()


def centres(layout, group):
    return {(round(x, 2), round(y, 2)) for x, y in layout.placed(group)}


print("%-14s %20s %8s" % ("group", type(stock).__name__, type(wide).__name__))
for group in mr.GROUPS:
    a, b = stock.placed(group), wide.placed(group)
    print("%-14s %20d %8d  %s"
          % (group, len(a), len(b), "" if len(a) == len(b) else "<-- changed"))
print("%-14s %20d %8d" % ("total", len(stock), len(wide)))

# Both thumbs end up with the same pair, one cluster the mirror of the other.
left = {(-x, y) for x, y in centres(wide, "left_thumb")}
print("\nthumbs mirror:", left == centres(wide, "right_thumb"))

# The centre row comes straight from Gccmx; the pinky is the one FgcLeverless drops.
print("centre is Crane's row:", wide.misc_coords == mr.Gccmx.misc_coords)
dropped, = centres(stock, "left_homerow") - centres(wide, "left_homerow")
print("left pinky dropped at: (%.1f, %.1f)" % dropped)

# Nothing came off the right hand, so the cluster no longer sits centred.
xs = [point[0] for point in wide.layout]
print("cluster centre now at x = %+.1fmm" % ((min(xs) + max(xs)) / 2))


group            CircleCapLeverless   WideGc
left_homerow                      4        3  <-- changed
right_homerow                     8        8  
left_thumb                        2        2  
right_thumb                       5        2  <-- changed
misc                              1        3  <-- changed
total                            20       18

thumbs mirror: True
centre is Crane's row: True
left pinky dropped at: (-153.3, 16.6)
cluster centre now at x = +13.3mm


## 2. Assemblies

Two stacks ship with the library, differing only in the faceplate: `f1CapMako1`
takes round Frame 1 caps and `mako1` takes square MX keycaps. The FGC build is
`f1CapMako1` on a smaller plate with the FGC layout — the configuration that
produced `notebooks/mako1_fgc/`.

`assemble()` stacks the layers along Z and colours them. `explode=True` loads the
viewer's explode tool — drag the animation slider under the render to pull the
plates apart and check that cutouts line up between layers.

In [4]:
fgc = mr.FgcLeverless()
fgc.layout_affine = Affine.translation(0, -13)

ASSEMBLIES = {
    "f1CapMako1": mr.f1CapMako1.set(
        width=in2mm(14), height=in2mm(6), layout=mr.CircleCapLeverless(),
    ),
    "mako1": mr.mako1.set(
        width=in2mm(14), height=in2mm(6), layout=mr.SquareCapLeverless(),
    ),
    "f1CapMako1 (FGC)": mr.f1CapMako1.set(
        width=in2mm(11.8), height=in2mm(5), hole_deltax=300, n_modelu=2, layout=fgc,
    ),
}

for name, built in ASSEMBLIES.items():
    print(name, "->", [type(part).__name__ for part in built])
    show(built.assemble(), names=[name], explode=True)

f1CapMako1 -> ['Base', 'WireSpaceModelU', 'Switchplate', 'F1CapFaceplate', 'Backplate']
+++cc


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

mako1 -> ['Base', 'WireSpaceModelU', 'Switchplate', 'KeycapFaceplate', 'Backplate']
ccccc


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

f1CapMako1 (FGC) -> ['Base', 'WireSpaceModelU', 'Switchplate', 'F1CapFaceplate', 'Backplate']
++++c


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

## 3. Housing a Brook board

`brookMako1` is the same five layers as `f1CapMako1` with two of them swapped,
for a build that carries a Brook Gen-5X/UFB. The board hangs face-up under the
switchplate on standoffs, and the rest follows from that: the wire space has to
be deep enough to hold it, and whatever its components reach above the
standoffs has to go up through a window in the switchplate.

Those numbers pull against each other, so the stack measures them rather than
assuming. `checks()` reports each one; `check()` raises instead, and the CLI
runs it before writing any DXF.

In [5]:
brook = mr.brookMako1
space = brook.part(mr.BrookWireSpace)
switch = brook.part(mr.BrookSwitchplate)

for ok, why in brook.checks():
    print("ok " if ok else "BAD", why)

print("\nstack is %.2fmm tall, %d layers" % (brook.height(), len(brook)))

ok  wire space 12.70mm (2 x 6.35mm stock) hangs the board on 8.10mm standoffs
ok  components stand 2.90mm proud of the switchplate's underside, within its 3.17mm thickness (3 spacers would clear it entirely)
ok  the switchplate is opened up over the board, which needs 2.90mm of relief
ok  window leaves 4.90mm of material between its edge and each screw
ok  wire space and switchplate agree on the board and where it sits
ok  board clears the nearest switch mount by 3.2mm
ok  board clears the nearest corner screw by 11.2mm
ok  board sits 18.7mm inside the cavity wall
ok  board clears the nearest corner pad by 104.5mm
ok  every board screw clears a switch mount, the tightest by 8.1mm

stack is 25.40mm tall, 5 layers


The three numbers you need to actually order parts, and where they come from.

In [6]:
board = space.board

print("board          %.2f x %.2fmm, %.1fmm thick" % (*board.size, board.thickness))
print("hole pattern   %.2f x %.2fmm  (%.1fmm in from each corner)"
      % (*board.hole_pitch(), board.hole_inset))
print("holes at       %s"
      % [(round(x, 2), round(y, 2)) for x, y in switch.board_hole_points()])
print()
print("wire space     %.2fmm  = %d x %.2fmm stock" % (space.part_depth(), space.spacers(), space.depth))
print("standoffs      %.2fmm  <- buy these" % space.standoff_length())
print("relief window  %.2f x %.2fmm, %.2fmm deep (switchplate is %.3fmm)"
      % (*switch.relief_size(), space.relief_depth(), switch.depth))
print("%d spacers would swallow the board whole, window and all"
      % space.spacers_without_relief())

board          96.01 x 45.01mm, 1.6mm thick
hole pattern   89.01 x 38.01mm  (3.5mm in from each corner)
holes at       [(-44.51, -41.0), (-44.51, -3.0), (44.51, -41.0), (44.51, -3.0)]

wire space     12.70mm  = 2 x 6.35mm stock
standoffs      8.10mm  <- buy these
relief window  76.01 x 25.01mm, 2.90mm deep (switchplate is 3.175mm)
3 spacers would swallow the board whole, window and all


Both the hole pattern and the component height are **assumptions** — Brook
publishes neither. The pattern is the one `SanwaBody` already assumed, so the
two stacks agree and a test keeps them that way. Measure your board before
cutting; the component height is what decides whether the faceplate still
closes over it.

A taller board is caught rather than cut:

In [7]:
tall = mr.brookMako1.set(board=mr.BrookBoard(component_height=30.0))

for ok, why in tall.checks():
    if not ok:
        print("BAD", why)

try:
    tall.check()
except ValueError as exc:
    print("\ncheck() raises:", exc)

BAD components stand 21.90mm proud of the switchplate's underside, within its 3.17mm thickness (6 spacers would clear it entirely)

check() raises: fit check failed: components stand 21.90mm proud of the switchplate's underside, within its 3.17mm thickness (6 spacers would clear it entirely)


The switchplate is where both changes show: four screw holes on the board's
pattern, and the window pulled in from the PCB outline by `board_rim` so a rim
is left carrying them.

In [8]:
print("switchplate: window %.2f x %.2fmm, %.2fmm of rim out to each screw"
      % (*switch.relief_size(), switch.rim_to_hole()))

show(switch.generate(), names=["BrookSwitchplate"])
show(brook.assemble(), names=["brookMako1"], explode=True)

switchplate: window 76.01 x 25.01mm, 4.90mm of rim out to each screw
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

c+ccc


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

## 4. A Sanwa build

`sanwaFgcPlus` is the FGC layout built out of arcade parts and opened out to a
two-handed board: 24mm snap-in buttons instead of mechanical switches and the
hands 200mm apart as in the smash layouts.

It is arranged around the **top** row of the right-hand eight being the home
row. The left hand mirrors the right hand's second row — four buttons, pinky
included — carried up a row so it lands level with that top row, and the thumbs
sit exactly where the smash layouts put MX and MY, just inboard of the index
finger and well below it. The three menu buttons are on the back panel rather
than the face, which lets the cluster sit square on the plate, and the board
goes under the right palm.

The layout has to open up to take arcade buttons — 22.5mm caps sit happily on a
26mm pitch, but a 24mm button has a 27mm bezel — so it scales the whole cluster
until the closest pair of centres is 30mm apart, then centres the bezels on the
plate. That, plus rear buttons whose bodies reach 24.5mm into the case, is what
makes this a 460 x 215mm board.

Instead of five cut plates it is three cut plates and one printed body. Nearly
every dimension is pinned by another one, so the assembly carries its constraints
as checks: `checks()` reports them, `check()` raises.

In [9]:
sanwa = mr.sanwaFgcPlus
plate = sanwa.part(mr.SanwaTopPlate)

# for ok, why in sanwa.checks():
#     print(" ok " if ok else "BAD ", why)

# print()
# print("%s buttons on a %.0f x %.0f x %.0fmm stack"
#       % (len(plate.layout), plate.width, plate.height, sanwa.height()))
# print(" -> ", [type(part).__name__ for part in sanwa])

Every plate fastens the same way, so the whole controller comes down to one
short list. The bottom plate takes a subset of the rim rather than repeating it
— nothing pushes on it — and each corner takes a single screw on its chamfer
rather than one either side.

In [10]:
# What it takes to bolt together: nothing threads into plastic, so the printed
# body gets brass heat-set inserts and every screw is an M3 into one of those.
for count, what in mr.sanwaFgcPlus.fasteners():
    print("%2d x %s" % (count, what))

26 x M3 heat-set inserts, 4.2mm hole x 6mm deep
15 x M3 x 14 screws, down through both top plates into the body
11 x M3 x 8 screws, up through the bottom plate
 4 x M3 x 8 self-tapping screws, up into the board standoffs
 2 x 4mm dowels x 10mm, locating the printed halves


`assemble()` stacks the four parts along Z. The plates are flat and go to a
laser — the bottom one is plain acrylic — and the body is the only part that has
to be printed.

In [11]:
show(sanwa.assemble(), names=["sanwaFgcPlus"], explode=True)

++++


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

The body is where everything that is not flat ends up: walls, the screw bosses
every plate fastens into, a slab under the lid that the Brook hangs from and that
takes two of the top plates' screws, a skirt turned inwards round the open bottom
to stiffen it and carry the bottom plate, and the back panel — USB-C on the
centreline in a Neutrik D-series footprint, with 24mm buttons divided evenly
either side of it, each with a slot either side so its snap tabs can be pinched
and the button pushed back out.

Tipped up here so the back wall faces the camera; the flat SVG fallback always
looks straight down Z.

In [12]:
body = sanwa.part(mr.SanwaBody)

print("back wall: jack at %.0f, 24mm buttons at %s"
      % (body.usb_offset, [round(x, 1) for x in body.rear_button_points()]))
print("release slots: %.1f x %.1fmm either side of each button"
      % (body.tab_slot_width, body.tab_slot_height))
print("screws along that edge:", [round(x, 1) for x in body.rear_screw_points()])
print("skirt: %.0fmm wide, %.1fmm thick" % (body.skirt_width, body.skirt_depth))

show(body.generate().rotate((0, 0, 0), (1, 0, 0), 90),
     names=["SanwaBody, seen from the back"])

back wall: jack at 0, 24mm buttons at [-82.0, -36.0, 36.0, 82.0]
release slots: 3.0 x 12.0mm either side of each button
screws along that edge: [-151.8, -59.0, 59.0, 151.8]
skirt: 16mm wide, 3.5mm thick
+


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

Set `rear_button_count=6` for three buttons a side. The pitch closes up on its
own so the outermost still lands on the straight part of the wall, and the screws
along that edge move into whatever bays are left.

In [13]:
six = sanwa.set(rear_button_count=6)
wall = six.part(mr.SanwaBody)

print("buttons at", [round(x, 1) for x in wall.rear_button_points()])
print("screws  at", [round(x, 1) for x in wall.rear_screw_points()])
print("all checks pass:", all(ok for ok, _ in six.checks()))

buttons at [-128.0, -82.0, -36.0, 36.0, 82.0, 128.0]
screws  at [-174.8, -105.0, -59.0, 59.0, 105.0, 174.8]
all checks pass: True


At 460 x 215mm the body does not fit a printer, so `halves()` cuts it in two.
The split does not run down the middle — the jack is there — but through the
clear bay beside it, and it works out where that bay is rather than being told.
The two pieces locate on dowels through the front and back walls, and the plates
screw into both of them.

In [14]:
left, right = body.halves()

print("split at x = %.1f" % body.split_x())
for name, half in [("left", left), ("right", right)]:
    box = half.val().BoundingBox()
    print("%-5s %6.1f x %5.1f x %4.1f mm" % (name, box.xlen, box.ylen, box.zlen))
print("bed needed: %.0f x %.0f mm" % body.print_bed)

show(left, right, names=["left", "right"])

split at x = 17.0
left   247.0 x 215.0 x 36.0 mm
right  213.0 x 215.0 x 36.0 mm
bed needed: 250 x 250 mm
++


CadViewerWidget(anchor=None, aspect_ratio=0.75, cad_width=800, control='trackball', glass=True, height=440, id…

## 5. Export for cutting

DXF wants the flat outline, so export the top face edges of each layer.

In [15]:
import pathlib

out = pathlib.Path("build")
out.mkdir(exist_ok=True)

for i, solid in enumerate(ASSEMBLIES["f1CapMako1 (FGC)"].generate()):
    exporters.export(solid.edges(">Z"), str(out / f"layer{i}.dxf"))

sorted(p.name for p in out.glob("*.dxf"))

['layer0.dxf', 'layer1.dxf', 'layer2.dxf', 'layer3.dxf', 'layer4.dxf']

The same thing from a shell, without the notebook:

```bash
python -m makoreactor --layout fgc --width 11.8 --height 5 -o build/
```

```bash
python -m makoreactor --stack sanwa -o build/
```

The Sanwa stack writes its plates as DXF and the body as STEP, since a flat
projection of a printed part is not much use.
